# Observability and Caching


Up to now, this project can ingest papers, index chunks, retrieve context, and answer questions through a local RAG pipeline. In this observability and caching phase, we add two production concerns on top of that working pipeline:

1. **Redis response caching** so repeated questions can skip retrieval and LLM generation.
2. **Langfuse observability** so each RAG request can be traced, timed, debugged, and analyzed.


## 0. Mental Model

So far, the application has this path:

```text
question -> retrieve chunks -> build prompt -> call Ollama -> answer
```

In this phase, we want to change it into:

```text
question -> build cache key -> check Redis
         -> cache hit: return stored answer quickly
         -> cache miss: retrieve -> prompt -> LLM -> store answer -> return
         -> trace each step in Langfuse
```

The core production idea is simple: expensive work should be observable, and repeated expensive work should usually be cached.

## 1. Why Caching Matters in RAG

A RAG answer has multiple costs:

- query embedding cost or latency
- vector/BM25 retrieval latency
- prompt construction latency
- LLM generation latency
- CPU, memory, or API cost

In this project, Ollama generation is usually the slowest part. A repeated question can take several seconds even though the final answer is identical or close enough to reuse.

A Redis response cache stores the full structured RAG result, not just the text answer. That means cached responses can include:

- answer
- sources
- search mode
- number of chunks
- timing metadata

The first request is a cache miss. The second identical request should be a cache hit.

## 2. Why Observability Matters in RAG

Normal logging tells us what happened at a coarse level. RAG observability should tell us how the answer was produced.

A useful trace should answer questions like:

- Did the request hit the cache?
- Which query was sent to retrieval?
- Which chunks were selected?
- How long did retrieval take?
- How long did generation take?
- Which model generated the answer?
- Did the model use the retrieved evidence well?
- Where did errors occur?

Langfuse is a tracing and analytics platform built for LLM applications. For this project, it should wrap the RAG pipeline with spans for cache, retrieval, prompt building, and generation.

## 3. Setup for Notebook Checks

These imports are only for notebook diagnostics. The real implementation later should live in `src/`, not inside this notebook.

In [41]:
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
import time
from pathlib import Path
from pprint import pprint

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None
import requests


def load_project_env() -> Path | None:
    """Load the nearest project .env without printing secrets."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        env_path = base / ".env"
        if not env_path.exists():
            continue

        if load_dotenv is not None:
            load_dotenv(env_path, override=False)
        else:
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
        return env_path
    return None


ENV_PATH = load_project_env()
print("Loaded .env:", ENV_PATH if ENV_PATH else "not found")

API_BASE = "http://localhost:8000"
ASK_URL = f"{API_BASE}/api/v1/ask/"
HYBRID_SEARCH_URL = f"{API_BASE}/api/v1/hybrid-search/"
HEALTH_URL = f"{API_BASE}/health"

Loaded .env: /Users/shishirrijal/Desktop/Projects/arxiv-rag-curator/.env


## 4. Baseline Health Check

Before adding Redis or Langfuse, verify the existing RAG services still work.

In [42]:
def get_json(url: str, timeout: int = 10) -> dict:
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()


try:
    health = get_json(HEALTH_URL)
    pprint(health)
except Exception as exc:
    print(f"API health check failed: {exc}")
    print("Start services with: docker compose up --build -d")

{'services': {'embeddings': {'fallback': 'bm25',
                             'reason': None,
                             'status': 'enabled'},
              'ollama': {'available_models': ['llama3.2:1b'],
                         'model': 'llama3.2:1b',
                         'model_ready': True,
                         'status': 'healthy'},
              'opensearch': {'cluster_status': 'green', 'status': 'healthy'},
              'postgresql': {'status': 'healthy'}},
 'status': 'healthy',
 'version': '0.4.0'}


## 5. Current API Surface

The observability and caching implementation will add cache-aware behavior and richer health data. This cell shows the actual API surface before that implementation.

In [43]:
try:
    openapi = get_json(f"{API_BASE}/openapi.json")
    paths = sorted(openapi.get("paths", {}).keys())
    print(f"Total endpoints: {len(paths)}")
    for path in paths:
        print(path)
except Exception as exc:
    print(f"Could not fetch OpenAPI schema: {exc}")

Total endpoints: 9
/
/api/v1/ask/
/api/v1/ask/stream
/api/v1/hybrid-search/
/api/v1/hybrid-search/health
/api/v1/search/
/api/v1/search/stats
/api/v1/search/{arxiv_id}
/health


## 6. Cache Keys: The Most Important Design Detail

A cache key decides whether two requests are considered the same.

For RAG, the key must include every parameter that can change the answer:

- question text
- hybrid vs BM25 mode
- category filters
- date filters
- model name, if model can vary
- prompt version, if prompts can change
- retrieval version, if retrieval logic can change

If the key is too broad, users get stale or wrong answers. If it is too narrow, cache hit rate drops.

In [44]:
def normalise_cache_payload(payload: dict) -> dict:
    """Make semantically identical request payloads produce stable JSON."""
    return {
        "question": payload.get("question", "").strip(),
        "use_hybrid": bool(payload.get("use_hybrid", True)),
        "categories": sorted(payload.get("categories") or []),
        "date_from": payload.get("date_from"),
        "date_to": payload.get("date_to"),
        "prompt_version": "rag-v1",
        "retrieval_version": "hybrid-v1",
    }


def make_cache_key(payload: dict) -> str:
    normalised = normalise_cache_payload(payload)
    raw = json.dumps(normalised, sort_keys=True, separators=(",", ":"))
    digest = hashlib.sha256(raw.encode("utf-8")).hexdigest()
    return f"rag:ask:{digest}"


payload_a = {"question": "What is the transformer architecture?", "use_hybrid": True}
payload_b = {"use_hybrid": True, "question": "What is the transformer architecture?"}
payload_c = {"question": "What is the transformer architecture?", "use_hybrid": False}

print(make_cache_key(payload_a))
print(make_cache_key(payload_b))
print(make_cache_key(payload_c))
print("A == B:", make_cache_key(payload_a) == make_cache_key(payload_b))
print("A == C:", make_cache_key(payload_a) == make_cache_key(payload_c))

rag:ask:6a6c3702d4714c2098f3a2316055bd639387d0662224905eaf17b912fc8cccf7
rag:ask:6a6c3702d4714c2098f3a2316055bd639387d0662224905eaf17b912fc8cccf7
rag:ask:2c7378360382d74c241c810237a843332d39fa574ed17f5cd34dd15d3d0efada
A == B: True
A == C: False


## 7. Exact Cache vs Semantic Cache

This phase starts with exact-match caching.

Exact caching means this hits:

```text
"What is the transformer architecture?"
"What is the transformer architecture?"
```

But this does not:

```text
"What is the transformer architecture?"
"Explain transformers"
```

Exact caching is simple, predictable, and safe. Semantic caching is more advanced: it embeds the query and reuses answers for similar questions. That can improve hit rate, but it requires careful thresholds because similar questions are not always equivalent.

## 8. Baseline Timing Without Redis Caching

Before implementing Redis, measure the current behavior. If the second request is still slow, that confirms there is no response cache yet.

In [45]:
def ask(question: str, use_hybrid: bool = True, timeout: int = 180) -> tuple[dict, float]:
    payload = {"question": question, "use_hybrid": use_hybrid}
    start = time.perf_counter()
    response = requests.post(ASK_URL, json=payload, timeout=timeout)
    elapsed = time.perf_counter() - start
    response.raise_for_status()
    return response.json(), elapsed


question = "What is the transformer architecture?"

try:
    first, first_seconds = ask(question)
    second, second_seconds = ask(question)

    print(f"First request:  {first_seconds:.2f}s")
    print(f"Second request: {second_seconds:.2f}s")
    print(f"First took_ms from API:  {first.get('took_ms')}")
    print(f"Second took_ms from API: {second.get('took_ms')}")
    print("Answers identical:", first.get("answer") == second.get("answer"))
except Exception as exc:
    print(f"Ask request failed: {exc}")

First request:  18.95s
Second request: 12.26s
First took_ms from API:  18926
Second took_ms from API: 12247
Answers identical: False


## 9. Notebook-Level Response Cache

The application does not have Redis caching wired into `/api/v1/ask/` yet. For learning, we can still build the same behavior in the notebook:

```text
ask_with_notebook_cache(payload)
  -> build stable cache key
  -> check Redis or in-memory fallback
  -> cache hit: return stored response
  -> cache miss: call real API, store response, return response
```

This is not the final architecture. It is a small local version of the exact cache logic we will later move into `src/`.

In [46]:
if importlib.util.find_spec("redis") is None:
    print("Installing redis client into this notebook environment...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "redis"])

import redis

CACHE_TTL_SECONDS = 24 * 60 * 60
_memory_cache: dict[str, tuple[float, str]] = {}


def make_redis_client():
    client = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)
    client.ping()
    return client


try:
    redis_client = make_redis_client()
    CACHE_BACKEND = "redis"
except Exception as exc:
    redis_client = None
    CACHE_BACKEND = "memory"
    print("Redis is not reachable from the notebook. Using in-memory fallback for the demo.")
    print(exc)

print("Cache backend:", CACHE_BACKEND)

Installing redis client into this notebook environment...



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Redis is not reachable from the notebook. Using in-memory fallback for the demo.
Error 61 connecting to localhost:6379. Connection refused.
Cache backend: memory


In [47]:
def cache_get(key: str) -> dict | None:
    if CACHE_BACKEND == "redis":
        raw = redis_client.get(key)
        return json.loads(raw) if raw else None

    entry = _memory_cache.get(key)
    if not entry:
        return None
    expires_at, raw = entry
    if expires_at < time.time():
        _memory_cache.pop(key, None)
        return None
    return json.loads(raw)


def cache_set(key: str, value: dict, ttl_seconds: int = CACHE_TTL_SECONDS) -> None:
    raw = json.dumps(value, sort_keys=True)
    if CACHE_BACKEND == "redis":
        redis_client.setex(key, ttl_seconds, raw)
        return
    _memory_cache[key] = (time.time() + ttl_seconds, raw)


def cache_delete(key: str) -> None:
    if CACHE_BACKEND == "redis":
        redis_client.delete(key)
        return
    _memory_cache.pop(key, None)


def ask_with_notebook_cache(payload: dict, bypass_cache: bool = False) -> tuple[dict, float]:
    cache_key = make_cache_key(payload)
    start = time.perf_counter()

    if not bypass_cache:
        cached = cache_get(cache_key)
        if cached is not None:
            elapsed = time.perf_counter() - start
            cached["cached"] = True
            cached["cache_backend"] = CACHE_BACKEND
            cached["cache_key"] = cache_key
            cached["took_ms"] = int(elapsed * 1000)
            return cached, elapsed

    response = requests.post(ASK_URL, json=payload, timeout=180)
    elapsed = time.perf_counter() - start
    response.raise_for_status()
    data = response.json()
    data["cached"] = False
    data["cache_backend"] = CACHE_BACKEND
    data["cache_key"] = cache_key
    data["original_api_took_ms"] = data.get("took_ms")
    cache_set(cache_key, data)
    return data, elapsed

## 10. Cache Miss vs Cache Hit Benchmark

Now we can reproduce the key cache test: first identical request should miss, second identical request should hit.

If the backend is Redis, the second response is coming from Redis. If Redis is not running, the same cache logic is demonstrated with an in-memory fallback.

In [48]:
cache_test_payload = {
    "question": "What is the transformer architecture?",
    "use_hybrid": True,
}
cache_delete(make_cache_key(cache_test_payload))

first_cached_response, first_cached_seconds = ask_with_notebook_cache(cache_test_payload)
second_cached_response, second_cached_seconds = ask_with_notebook_cache(cache_test_payload)

speedup = first_cached_seconds / max(second_cached_seconds, 0.001)

print("Backend:", CACHE_BACKEND)
print(f"First request:  {first_cached_seconds:.2f}s | cached={first_cached_response['cached']}")
print(f"Second request: {second_cached_seconds:.3f}s | cached={second_cached_response['cached']}")
print(f"Speedup: {speedup:.0f}x")
print("Same answer:", first_cached_response.get("answer") == second_cached_response.get("answer"))
print("Cache key:", second_cached_response["cache_key"])

Backend: memory
First request:  7.77s | cached=False
Second request: 0.000s | cached=True
Speedup: 7769x
Same answer: True
Cache key: rag:ask:6a6c3702d4714c2098f3a2316055bd639387d0662224905eaf17b912fc8cccf7


## 11. Different Query Should Miss

Exact-match caching should not reuse an answer for a different question. This is the conservative behavior we want first.

In [49]:
different_payload = {
    "question": "How transparent is DiffusionGemma?",
    "use_hybrid": True,
}
different_response, different_seconds = ask_with_notebook_cache(different_payload)

print(f"Different query time: {different_seconds:.2f}s")
print("cached:", different_response["cached"])
print("cache key differs:", different_response["cache_key"] != second_cached_response["cache_key"])
print("sources:", [s.get("arxiv_id") for s in different_response.get("sources", [])])

Different query time: 13.38s
cached: False
cache key differs: True
sources: ['2606.20560']


## 12. Cache Invalidation

Caching needs an escape hatch. For this notebook demo, invalidation is just deleting the exact cache key.

In the application later, invalidation can happen by TTL, by manual delete, or by changing the cache key version when prompts/retrieval logic/corpus contents change.

In [50]:
key_to_delete = make_cache_key(cache_test_payload)
print("Before delete exists:", cache_get(key_to_delete) is not None)
cache_delete(key_to_delete)
print("After delete exists:", cache_get(key_to_delete) is not None)

Before delete exists: True
After delete exists: False


## 13. What Redis Will Store

For this project, the Redis value should be JSON that matches the API response.

Example shape:

```json
{
  "question": "What is the transformer architecture?",
  "answer": "...",
  "sources": [...],
  "search_mode": "hybrid",
  "n_chunks": 5,
  "took_ms": 6123,
  "cached": false
}
```

When returning from cache, the API should usually update metadata such as:

- `cached: true`
- `took_ms`: time to retrieve from Redis, not original LLM time

The stored value should not include secrets, raw prompts with sensitive user data, or internal stack traces.

## 14. Redis Health Check Target

After implementation, the main `/health` response should include Redis.

Target shape:

```json
{
  "services": {
    "redis": {
      "status": "healthy",
      "ttl_hours": 24
    }
  }
}
```

A failing Redis should probably degrade the app, not break it. The system can still answer without cache; it will just be slower.

In [51]:
try:
    health = get_json(HEALTH_URL)
    redis_health = health.get("services", {}).get("redis")
    if redis_health:
        pprint(redis_health)
    else:
        print("Redis is not currently exposed in /health. This is expected before the caching implementation.")
except Exception as exc:
    print(f"Health request failed: {exc}")

Redis is not currently exposed in /health. This is expected before the caching implementation.


## 15. Langfuse Trace Design

A good RAG trace should have one parent trace per user request and child spans/generations for each expensive step.

Target trace structure:

```text
trace: rag.ask
  span: cache.lookup
  span: retrieval.hybrid_search
  span: context.build_prompt
  generation: ollama.generate
  span: cache.store
```

For a cache hit:

```text
trace: rag.ask
  span: cache.lookup
  output: cached response
```

This is the difference between knowing that a request was slow and knowing why it was slow.

## 16. Langfuse Health Check Target

Langfuse can run as a cloud service or self-hosted service. For this local learning project, either is acceptable.

Environment variables usually include:

```text
LANGFUSE_SECRET_KEY=...
LANGFUSE_PUBLIC_KEY=...
LANGFUSE_HOST=https://cloud.langfuse.com
```

For self-hosted Langfuse, the host might be `http://langfuse:3000` inside Docker and `http://localhost:3000` from the host machine.

For production behavior, Langfuse failure should not break question answering. Observability is important, but the user-facing path should degrade gracefully.

In [52]:
LANGFUSE_LOCAL_HEALTH = "http://localhost:3000/api/public/health"

try:
    response = requests.get(LANGFUSE_LOCAL_HEALTH, timeout=5)
    print("Langfuse status code:", response.status_code)
    print(response.text[:300])
except Exception as exc:
    print("Langfuse is not reachable locally yet. This is expected before the observability implementation.")
    print(exc)

Langfuse is not reachable locally yet. This is expected before the observability implementation.
HTTPConnectionPool(host='localhost', port=3000): Max retries exceeded with url: /api/public/health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=3000): Failed to establish a new connection: [Errno 61] Connection refused"))


## 17. Tiny Langfuse Demo

This section sends one small synthetic RAG trace to Langfuse so you can see what observability looks like before wiring it into the real API.

It does not call your RAG endpoint. It creates a trace with the same shape we eventually want in `src/`:

```text
trace: notebook.rag_demo
  span: cache.lookup
  span: retrieval.hybrid_search
  generation: ollama.generate
  span: cache.store
```

The cell reads Langfuse keys from `.env` and does not print secrets. It uses the current Langfuse Python SDK style: a client plus nested observations.

In [53]:

from langfuse import get_client
import os

# Make this cell safe to run even if the setup cell was not rerun.
try:
    ENV_PATH = load_project_env()
except NameError:
    from pathlib import Path
    try:
        from dotenv import load_dotenv
    except ImportError:
        load_dotenv = None

    ENV_PATH = None
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / ".env"
        if candidate.exists():
            ENV_PATH = candidate
            if load_dotenv is not None:
                load_dotenv(candidate, override=False)
            else:
                for line in candidate.read_text().splitlines():
                    line = line.strip()
                    if not line or line.startswith("#") or "=" not in line:
                        continue
                    key, value = line.split("=", 1)
                    os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
            break

print("Loaded .env:", ENV_PATH if ENV_PATH else "not found")

public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
secret_key = os.getenv("LANGFUSE_SECRET_KEY")
langfuse_host = os.getenv("LANGFUSE_BASE_URL") or os.getenv("LANGFUSE_HOST") or "https://cloud.langfuse.com"
os.environ.setdefault("LANGFUSE_HOST", langfuse_host)

missing = [
    name
    for name, value in {
        "LANGFUSE_PUBLIC_KEY": public_key,
        "LANGFUSE_SECRET_KEY": secret_key,
    }.items()
    if not value
]

if missing:
    raise RuntimeError(f"Missing Langfuse environment variables: {missing}")

langfuse = get_client()

if not langfuse.auth_check():
    raise RuntimeError("Langfuse authentication failed. Check keys and LANGFUSE_BASE_URL in .env.")

print("Langfuse client authenticated")
print("Host:", langfuse_host)
print("Public key prefix:", public_key[:8] + "...")

Loaded .env: /Users/shishirrijal/Desktop/Projects/arxiv-rag-curator/.env
Langfuse client authenticated
Host: https://us.cloud.langfuse.com
Public key prefix: pk-lf-88...


Now create a single trace. This is fake pipeline data, but the structure is the important part.

When this is later moved into `src/`, each span will wrap a real operation: cache lookup, hybrid retrieval, prompt construction, Ollama generation, and cache storage.

In [54]:
question = "What is the transformer architecture?"

answer = "The Transformer is a sequence model architecture based on attention mechanisms instead of recurrence or convolution."

with langfuse.start_as_current_observation(
    as_type="span",
    name="notebook.rag_demo",
) as root:
    root.update(input={"question": question, "use_hybrid": True})

    with langfuse.start_as_current_observation(
        as_type="span",
        name="cache.lookup",
    ) as cache_span:
        cache_span.update(input={"cache_key": make_cache_key({"question": question, "use_hybrid": True})})
        time.sleep(0.02)
        cache_span.update(output={"hit": False})

    with langfuse.start_as_current_observation(
        as_type="span",
        name="retrieval.hybrid_search",
    ) as retrieval_span:
        retrieval_span.update(input={"query": question, "top_k": 5})
        time.sleep(0.05)
        retrieval_span.update(output={
            "search_mode": "hybrid",
            "chunks": [
                {
                    "arxiv_id": "1706.03762",
                    "title": "Attention Is All You Need",
                    "section": "Model Architecture",
                }
            ],
        })

    with langfuse.start_as_current_observation(
        as_type="generation",
        name="ollama.generate",
        model="llama3.2:1b",
    ) as generation:
        generation.update(input={
            "prompt_preview": "Context: Attention Is All You Need...\nQuestion: What is the transformer architecture?",
        })
        time.sleep(0.1)
        generation.update(output=answer)

    with langfuse.start_as_current_observation(
        as_type="span",
        name="cache.store",
    ) as store_span:
        time.sleep(0.02)
        store_span.update(output={"stored": True, "ttl_hours": 24})

    root.update(output={
        "answer": answer,
        "sources": ["1706.03762"],
        "cached": False,
    })

langfuse.flush()

print("Trace sent to Langfuse")
print("Trace name: notebook.rag_demo")
print("Open your Langfuse dashboard and look under Traces.")

Trace sent to Langfuse
Trace name: notebook.rag_demo
Open your Langfuse dashboard and look under Traces.


## 18. Trace a Real API Call

The previous demo created fake pipeline steps to show the shape of a RAG trace. Now we can wrap a real call to `/api/v1/ask/`.

This gives request-level observability: status code, total API latency, answer preview, source metadata, search mode, and chunk count.

It does not yet show internal spans for retrieval, prompt building, or Ollama generation. To see those, Langfuse needs to be added inside `src/services/rag/pipeline.py` later.

In [55]:
real_question = "What is the transformer architecture?"
request_payload = {"question": real_question, "use_hybrid": True}

with langfuse.start_as_current_observation(
    as_type="span",
    name="notebook.real_api_call",
) as api_trace:
    api_trace.update(input={
        "url": ASK_URL,
        "payload": request_payload,
    })

    start = time.perf_counter()
    response = requests.post(ASK_URL, json=request_payload, timeout=180)
    elapsed_ms = int((time.perf_counter() - start) * 1000)

    try:
        data = response.json()
    except ValueError:
        data = {"raw_response": response.text[:500]}

    api_trace.update(output={
        "status_code": response.status_code,
        "elapsed_ms": elapsed_ms,
        "api_took_ms": data.get("took_ms"),
        "search_mode": data.get("search_mode"),
        "n_chunks": data.get("n_chunks"),
        "sources": data.get("sources", []),
        "answer_preview": data.get("answer", "")[:400],
    })

langfuse.flush()

print("Real API trace sent to Langfuse")
print("Trace name: notebook.real_api_call")
print("Status:", response.status_code)
print("Elapsed:", elapsed_ms, "ms")
print("API took_ms:", data.get("took_ms"))
print("Sources:", [source.get("arxiv_id") for source in data.get("sources", [])])

Real API trace sent to Langfuse
Trace name: notebook.real_api_call
Status: 200
Elapsed: 9656 ms
API took_ms: 9641
Sources: ['1706.03762', '2606.20560']


## 19. Trace Cache Hit and Miss Around the Real API

Now combine both ideas: use the notebook-level cache wrapper and send Langfuse traces that show whether the request was a cache miss or cache hit.

This is the closest notebook-only version of the final app behavior. The first trace should include an API call and cache store. The second trace should skip the API call because the response is already cached.

In [56]:
traced_payload = {
    "question": "What is the transformer architecture?",
    "use_hybrid": True,
}
traced_cache_key = make_cache_key(traced_payload)
cache_delete(traced_cache_key)


def traced_cached_api_call(payload: dict, trace_name: str) -> dict:
    cache_key = make_cache_key(payload)
    with langfuse.start_as_current_observation(as_type="span", name=trace_name) as root:
        root.update(input={"payload": payload, "cache_key": cache_key})

        with langfuse.start_as_current_observation(as_type="span", name="cache.lookup") as cache_span:
            cached = cache_get(cache_key)
            cache_span.update(input={"cache_key": cache_key}, output={"hit": cached is not None})

        if cached is not None:
            cached["cached"] = True
            root.update(output={
                "cached": True,
                "status_code": 200,
                "answer_preview": cached.get("answer", "")[:300],
                "sources": cached.get("sources", []),
            })
            return cached

        with langfuse.start_as_current_observation(as_type="span", name="api.post_ask") as api_span:
            start = time.perf_counter()
            response = requests.post(ASK_URL, json=payload, timeout=180)
            elapsed_ms = int((time.perf_counter() - start) * 1000)
            response.raise_for_status()
            data = response.json()
            api_span.update(input={"url": ASK_URL}, output={
                "status_code": response.status_code,
                "elapsed_ms": elapsed_ms,
                "api_took_ms": data.get("took_ms"),
                "n_chunks": data.get("n_chunks"),
                "search_mode": data.get("search_mode"),
            })

        data["cached"] = False
        data["cache_key"] = cache_key
        data["cache_backend"] = CACHE_BACKEND

        with langfuse.start_as_current_observation(as_type="span", name="cache.store") as store_span:
            cache_set(cache_key, data)
            store_span.update(output={"stored": True, "ttl_seconds": CACHE_TTL_SECONDS})

        root.update(output={
            "cached": False,
            "status_code": response.status_code,
            "answer_preview": data.get("answer", "")[:300],
            "sources": data.get("sources", []),
        })
        return data


miss_result = traced_cached_api_call(traced_payload, "notebook.cached_api_call.miss")
hit_result = traced_cached_api_call(traced_payload, "notebook.cached_api_call.hit")
langfuse.flush()

print("Cache miss trace sent:", not miss_result["cached"])
print("Cache hit trace sent:", hit_result["cached"])
print("Look for traces: notebook.cached_api_call.miss and notebook.cached_api_call.hit")

Cache miss trace sent: True
Cache hit trace sent: True
Look for traces: notebook.cached_api_call.miss and notebook.cached_api_call.hit


## 20. What We Should Implement Later in `src/`

After this notebook is understood, the source implementation should be small and staged.

Suggested commits for branch `feature/observability-caching`:

1. Add Redis and Langfuse settings plus dependencies.
2. Add Redis service with `get`, `set`, `delete`, `health_check`, and stable cache key helpers.
3. Add cache check/store around the non-streaming `/ask` pipeline.
4. Add Langfuse service wrapper with graceful no-op behavior when credentials are missing.
5. Instrument RAG steps: cache, retrieval, context building, generation.
6. Expose cache/observability status in `/health`.
7. Update Gradio to display cache hit and response time if the API exposes that metadata.

For streaming, be careful: caching full streaming responses is possible, but the implementation is different because the answer arrives token by token. Start with non-streaming `/ask` first.

## 21. Production Tradeoffs

Caching is not free. It introduces correctness tradeoffs.

Questions to answer before implementation:

- Should cache be invalidated when new papers are ingested?
- Should cache keys include corpus version or latest index timestamp?
- Should users be able to bypass cache?
- Should low-confidence or refusal answers be cached?
- Should streaming and non-streaming endpoints share cache entries?
- Should cache entries include model name and prompt version?

For this learning project, the pragmatic first version is exact-match caching with a TTL and prompt/retrieval version fields in the key.

## 22. Summary

This phase is not about changing the retrieval algorithm. It is about operating the RAG system like a production service.

You should understand these before writing `src/` code:

- Redis is for avoiding repeated expensive RAG work.
- Exact cache keys must include all answer-changing parameters.
- TTL prevents stale answers from living forever.
- Langfuse traces make the pipeline inspectable request by request.
- Cache and observability failures should degrade gracefully.

The next phase is to implement these concepts in the application code, starting with Redis cache support for the non-streaming `/api/v1/ask/` endpoint.